# Heart Disease Prediction by using ML
## 1. Problem Definition 
Predict whether individual has heart disease or not according given features
## 2. Data Introduction
* Data was obtained from [Kaggle](https://www.kaggle.com/datasets/sumaiyatasmeem/heart-disease-classification-dataset).

## 3. Features
We've got 13 features, and 1 target in the dataset, and it's described as below.

1. **age** - age in years

2. **sex** - gender
    * 1 : male
    * 0 : female

3. **cp** - Chest pain type
    * 0 : typical angina
    * 1 : atypical angina
    * 2 : non — anginal pain
    * 3 : asymptotic

4. **trestbps**- resting blood pressure in mmHg (unit)
    * anything above 130-140 is typically cause for concern.

5. **chol**- Serum Cholestrol in mg/dl (unit)

6. **fbs**- Fasting Blood Sugar
    * fasting blood sugar > 120mg/dl
        * 1 = true
        * 0 = false
    * '>126' mg/dL represents diabetes

7. **restecg** - resting electrocardiographic
    * 0 : normal
    * 1 : having ST-T wave abnormality
    * 2 : left ventricular hyperthrophy

8. **thalach**- maximum heart rate

9. **exang**- Exercise induced angina
    * 1 : yes
    * 0 : no

10. **oldpeak**- ST depression induced by exercise relative to rest looks at stress of heart during excercise unhealthy heart will stress more

11. **slope** - Slope of the peak exercise ST segment
    * 0 : upsloping -  better heart rate with excercise (uncommon)
    * 1 : flat -  minimal change (typical healthy heart)
    * 2 : downsloping - signs of unhealthy heart

12. **ca** - Number of major vessels (0–3) colored by flourosopy
    * colored vessel means the doctor can see the blood passing through
    * the more blood movement the better (no clots)

13. **thal** - Displays the thalassemia
    * 1,3 : normal
    * 6 : fixed defect
    * 7 : reversible defect (no proper blood movement when excercising)

14. **target** - A patient has heart diseas or not
    * 1 : yes
    * 0 : no

In [ ]:
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

# Models
from sklearn.svm import LinearSVC, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.model_selection import train_test_split

# metrics
from sklearn.metrics import  roc_curve, ConfusionMatrixDisplay, precision_score, RocCurveDisplay, classification_report, f1_score, recall_score, precision_score

## 4. EDA

In [ ]:
df = pd.read_csv('./data/heart-disease.csv')
df

In [ ]:
print(df.isna().sum())
print("===================================")
df.info()

This is a cleaned dataset

Checking the describe Statistics

In [ ]:
df.describe()

### 4.1 Checking if class balanced

In [ ]:
print(df["target"].value_counts())

df["target"].value_counts().plot(kind = "bar",
                                figsize = (5, 4),
                                color = ['pink', 'lightblue'])
plt.title("Class balanced or not")
plt.ylabel("counts");

#### Checking Heart Disease Frequency with Sex
0 = 'female'

1 = 'male'

In [ ]:
pd.crosstab(df['target'],df['sex'])

In [ ]:
print(df['sex'].value_counts())
pd.crosstab(df['target'], df['sex']).plot(kind = 'bar',
                             color = ['pink', 'lightblue'],
                             figsize = (5,4))
plt.title("Heart Disease Frequency given gender")
plt.legend(["Female", "Male"])
plt.xlabel("Having heart disease or not")
plt.ylabel("counts");

We have more males than females in the dataset. Among males, the difference between those with and without heart disease is relatively small. In contrast, heart disease frequency among females is quite imbalanced. The number of females with heart disease is more than three times that of those without. This imbalance may introduce bias when training models. we can keep this in mind.

What about age? How does age affects the heart disease development?

Let's check on the distribution of age in the dataset firstly.

In [ ]:
df['age'].plot.hist(bins = 7);

The majority of individuals in the dataset fall between the age of 55 and 65.

Let's check the relationship among age, maximum heart rate, and heart disease with a plot.

In [ ]:
fig, ax = plt.subplots(figsize = (10, 6))

ax.scatter(df.age[df.target == 0], df.thalach[df.target == 0], c = "deeppink")
ax.scatter(df.age[df.target == 1], df.thalach[df.target == 1], c = "lawngreen")

ax.set(title = "Hear Disease",
       xlabel = "Age (year)",
       ylabel = "Max Heart Rate")
ax.axhline(np.mean(df["thalach"]), c = 'blue', linestyle = "--")
ax.legend(["Disease", "No Disease"]);

The plot suggests that as age increases, the maximum heart rate tends to decrease. Furthermore, the likelihood of developing heart disease increases when the maximum heart rate is below average(There are more pink dots under the blue line).

Let's make correlation matrix to check which features have stronger relationsip with the target

In [ ]:
# correlation

df.corr()

In [ ]:
corr_matrix = df.corr()

fig, ax = plt.subplots(figsize = (15, 10))
ax = sns.heatmap(corr_matrix,
                annot = True,
                linewidth = 0.5,
                fmt = ".2f",
                cmap = "YlGnBu")

From the correlation matrix, several important relationshiops can be observed

The correlation matrix shows that **cp** and **thalach** have relatively strong positive correlations with the target, while **exang** and **oldpeak** exhibit relatively strong negative correlations.

### 5. Modelling

#### 5.1 Split the data into training and test set

In [ ]:
X = df.drop(['target'], axis = 1)
y = df['target'].to_numpy()

np.random.seed(42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2)

In [ ]:
# Checking the dataset

X_train.shape, X_test.shape, y_train.shape, y_test.shape
X_train

In [ ]:
y_train

#### 5.2 Model Selection

**I am trying the following ML models:**

Models are selected by referring to the [map](https://scikit-learn.org/stable/machine_learning_map.html). Additionally, since this is a binary classification problem, `Logistic Regression` and `Random Forest` are also considered.

1. LinearSVC
3. K-Nearest Neighbors Classifier
4. Logistic Regression
5. Random Forest Classifier

#### Evaluation Function
Before training models, creating an reusable evaluation function is easier to go.

In [ ]:
def model_eval(model, X_train, y_train, X_test, y_test):
    
    fig = plt.figure(figsize = (25, 8))   

    # Confusion Matrix - training dataset
    ax0 = fig.add_subplot(1, 3, 1)
    ConfusionMatrixDisplay.from_estimator(model, X_train, y_train, ax = ax0)
    ax0.set(title = f"Training set accuracy: {model.score(X_train, y_train) * 100:.2f}%")

    # Confusion Matrix - Test dataset
    ax1 = fig.add_subplot(1, 3, 2)
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, ax = ax1)
    ax1.set(title = f"Test set accuracy: {model.score(X_test, y_test)*100:.2f}%")

    # roc curve
    ax2 = fig.add_subplot(1,3,3)
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax = ax2)   

    # classification report
    print(classification_report(y_test, model.predict(X_test)))    

#### 5.3 Fit models

##### Linear SVC

In [ ]:
lsvc_clf = LinearSVC()
lsvc_clf.fit(X_train, y_train)
model_eval(lsvc_clf, X_train, y_train, X_test, y_test)

lsvc_clf.get_params()

Tuning Linear SVC by GridSearchCV

In [ ]:
param_grid = {"C": [0.01, 0.1, 1, 10, 100]}

lsvc_grid_cv = GridSearchCV(LinearSVC(max_iter = 5000),
                            param_grid,
                            cv =5)
lsvc_grid_cv.fit(X_train, y_train)  

model_eval(lsvc_grid_cv, X_train, y_train, X_test, y_test)
lsvc_grid_cv.best_params_

In our case, the main concern is failing to diagnose a patient who actually has heart disease. Therefore, we first focus on recall, which is 0.91, indicating strong performance. This is also reflected in the confusion matrix of the test set, where only one sample was misclassified as negative despite the individual having heart disease. In addition, the overall accuracy is slightly improved from approximately 86% to 88%.

However, since our data is slightly imbalanced, F1-score should also be taken into account.
The f1-score is slightly increased from 0.88 to 0.89 after applying the optimal parameter through grid search.

##### Further experiments will be conducted to assess the performance of other models:
* K-Nearest Neighbors Classifier
* Logistic Regression
* Random Forest Classifier

It will be performed in one go by applying `for loop`

In [ ]:
trained_models = {}

models = {'KNN' : KNeighborsClassifier(),
         'LogisticRegression' : LogisticRegression(),
         'RandomForest' : RandomForestClassifier()}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[model_name] = model

The iteration warning in Logistic Regression is usually caused by a combination of factors such as **insufficient iterations**, **lack of feature scaling**, **inappropriate solver choice**, and **data characteristics** like multicollinearity. In practice, I handle this by increasing max_iter

In [ ]:
trained_models = {}

models = {'KNN' : KNeighborsClassifier(),
         'LogisticRegression' : LogisticRegression(max_iter = 1000),
         'RandomForest' : RandomForestClassifier()}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[model_name] = model

### 6. Model Evaluation

In [ ]:
model_eval(trained_models['KNN'], X_train, y_train, X_test, y_test)

In [ ]:
model_eval(trained_models['LogisticRegression'], X_train, y_train, X_test, y_test)

In [ ]:
model_eval(trained_models['RandomForest'], X_train, y_train, X_test, y_test)

### Model comparison

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))

RocCurveDisplay.from_estimator(
    trained_models['LogisticRegression'], X_test, y_test,
    ax=ax,
    name="Logistic Regression",
    curve_kwargs = {"color": "blue"}
)

RocCurveDisplay.from_estimator(
    trained_models['RandomForest'], X_test, y_test,
    ax=ax,
    name="Random Forest",
    curve_kwargs = {"color": "green"}
)

RocCurveDisplay.from_estimator(
    trained_models['KNN'], X_test, y_test,
    ax=ax,
    name="KNN",
    curve_kwargs = {"color": "red"}
)

RocCurveDisplay.from_estimator(
    lsvc_grid_cv, X_test, y_test,
    ax=ax,
    name="Linear SVC",
    curve_kwargs = {"color": "purple"}
)



plt.title("ROC Curve Comparison")
plt.show();

The AUC plot indicates that **K-Nearest Neighbors** performs the worst, whereas the other models exhibit similar performance.

The ROC curve indicates that is the optimal model, as it achieves a high true positive rate while maintaining a low false positive rate. In heart disease classification, minimizing the false negative rate is critical, so maximizing the true positive rate is particularly important. However, the training accuracy of  **Random Forest** reaches 100%, which may indicate overfitting.

In [ ]:
# model_comparison

accuracy_dict = {}
f1_score_dict = {}
recall_dict = {}
precision_dict = {}

for model in 'KNN', 'LogisticRegression', 'RandomForest':
    accuracy_dict[model] = trained_models[model].score(X_test, y_test)

    y_pred = trained_models[model].predict(X_test)
    f1_score_dict[model] = f1_score(y_pred, y_test)
    recall_dict[model] = recall_score(y_pred, y_test)
    precision_dict[model] = precision_score(y_pred, y_test)
    

# LinearSVC metrics
accuracy_dict['LinearSVC'] = lsvc_clf.score(X_test, y_test)
y_pred = lsvc_clf.predict(X_test)
f1_score_dict['LinearSVC'] = f1_score(y_pred, y_test)
recall_dict['LinearSVC'] = recall_score(y_pred, y_test)
precision_dict['LinearSVC'] = precision_score(y_pred, y_test)

accr_compare = pd.DataFrame(accuracy_dict, index = ['accuracy'])
model_comparison = accr_compare.T
model_comparison['f1'] = f1_score_dict.values()
model_comparison['recall'] = recall_dict.values()
model_comparison['precision'] = precision_dict.values()
model_comparison

In [ ]:
model_comparison.plot(kind = 'bar',
                     figsize = (8, 6),
                     cmap = "plasma");

plt.title("Model metrics comparison")
plt.xlabel("Model")
plt.ylabel("Score")
plt.xticks(rotation = 45)
plt.tight_layout()
plt.show()

The plot indicates that LogisticRegression model performs the best whereas model K- Nearest Neighbors Classifier performs the worst, with an accuracy of around 68% only. 

Several intereting observations can be made. The recall scores of Linear SVC and Logistic Regression are nearly identical, and this similarity is also reflected in their AUC values. 

In brief, Logistic Regression and Linear SVC are our most promising models so far, given the poor performance of the K-Nearest Neighbors classifier. Furthermore, the Random Forest model may be overfitting.

#### Applying `Cross validation`
Cross-validation is applying in order to mitigate the risk of obtaining biased results from a single data split.

In [ ]:
np.random.seed(45)

models = { 'Linear SVC' : LinearSVC(),
        'KNN' : KNeighborsClassifier(),
         'LogisticRegression' : LogisticRegression(max_iter = 1000),
         'RandomForest' : RandomForestClassifier()}

models_acc = {}
for model_name, model in models.items():
    cv_acc = {}
    for score in "accuracy", "precision", "f1", "recall":
        cv_acc[score] = cross_val_score(model, X, y, cv = 5, scoring = score)
    models_acc[model_name] = cv_acc
models_acc

Average cross-validation metrics.

In [ ]:
for model, metrics in models_acc.items():
    print(model)
    for metric, accuracy_lst in metrics.items():
        print(f"{metric} avg: {np.mean(accuracy_lst)}")
    print()

In [ ]:
models_acc_avg = models_acc
for model, metrics in models_acc.items(): 
    for metric, accuracy_lst in metrics.items():        
        models_acc_avg[model][metric] = round(np.mean(accuracy_lst), 3)

pd.DataFrame(models_acc_avg).T

Visualisation

In [ ]:
pd.DataFrame(models_acc_avg).T.plot(kind = 'bar',
                                    figsize = (8,6),
                                    cmap = 'plasma',
                                   title = "Cross-validated classification mean metrics")
plt.legend(loc = 'lower right')
plt.xticks(rotation = 45)
plt.tight_layout()
plt.show();

After cross-validation, the average recall scores across models show a slight improvement, with Logistic Regression still performing the best. Therefore, we proceed with hyperparameter tuning to further enhance its performance.

### Hyper parameter tuning


We use `np.logspace(-4, 4, 30)` to explore C values across several orders of magnitude, because the effect of C on regularization strength is exponential rather than linear. This ensures that both very small and very large values are properly evaluated.

In [ ]:
log_reg_grid = {"C": np.logspace(-4, 4, 30),
                "solver": ["liblinear", "lbfgs"]}

gs_log_reg = GridSearchCV(LogisticRegression(max_iter = 2000),
                          param_grid=log_reg_grid,
                          cv=5,
                          verbose=True)

gs_log_reg.fit(X_train, y_train)

Check the best hyperparameters

In [ ]:
gs_log_reg.best_params_

In [ ]:
model_eval(gs_log_reg, X_train, y_train, X_test, y_test)

In [ ]:
# compare the metrics to our baseline Logistic Regression model
model_eval(trained_models['LogisticRegression'], X_train, y_train, X_test, y_test) # Our firest logisticregression model

Hypterparameter tuning with GridSearchCV did not lead to significant performance improvement for the Logistic Regression model.

### 7. Feature Importance

In [ ]:
logistic_clf = LogisticRegression(C = 0.20433597178569418, # applying the optimistic parameters by GridSearchCV
                                  solver = 'liblinear') 
logistic_clf.fit(X_train, y_train)

In [ ]:
logistic_clf.coef_

In [ ]:
# Match coef's of features to columns
feature_dict = dict(zip(df.columns, list(logistic_clf.coef_[0])))

feature_df = pd.DataFrame(feature_dict, columns = feature_dict.keys(),
            index = [0])
feature_df

In [ ]:
# plt.style.available
plt.style.use('seaborn-v0_8')

feature_df.T.plot.bar(title="Feature Importance",
                      figsize = (6, 5),
                      legend = False);

Feature Importance Explanation

#### Trying XGBoost model

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier()
xgb.fit(X_train, y_train)
model_eval(xgb, X_train, y_train, X_test, y_test)

The recall-score can be increased by decreasing the decision threshold (default is 0.5 [refer to sklearn document](https://scikit-learn.org/stable/modules/classification_threshold.html)) whereas the precision-score will be decrased.


# Reference
* [sklearn Docs](https://scikit-learn.org/stable/index.html)
* [Daniel Bourke](https://github.com/mrdbourke/zero-to-mastery-ml/blob/master/section-3-structured-data-projects/end-to-end-heart-disease-classification.ipynb)